# Reports of analysis on NBA player's performance and salary

## Introduction

The goal of this analysis is to determine whether a player's performance fits his salary and give suggestions on which players would be good fit for the team. The methodology used in this analysis is k-means clustering.

First of all, what is k-means? K-means is a clustering algorithm that categorizes data points into k clusters based on their feature. It is very useful when we want to group data points with similar patterns. In this analysis, I will choose the clusters using some stats that represent the player's performance. We will have n groups of players based on their performance (one group has good performance, one group has mediocre performance, and another group has bad performance, etc.) and we will compare their salary to see if they are overpaid, underpaid or fairly paid.

In the game, the determining factor of a player's performance is how he contributes to the scoring of the team. The simplest way to interpret it is to look at a players' scoring ability, including the productivity, accuracy, consistency of his scoring.

In analyzing the players' performance, I used "MP", "eFG%", and "PT" as the performance metrics. I pick PTS as it is the most direct contributor to a player's contribution. I chose eFG% because it takes the positional difference of players into account. Centers tend to have a higher percentage as they attack close to the rim, but not necessarily eFG%, as it is calculated using both 2PT and 3PT shots. Minutes Played is a good indicator of how much a player is used. A player who plays 30 minutes a game is likely to be more valuable than one who plays 10 minutes a game, even if they have the same PTS and eFG%.

I first standardized the stats to offset the impact of large numbers (e.g PTS) on the overall evaluation. Then I filtered out players who played less than 20 games and those who played for less than 10 minutes to acquire a more valid dataset. I then used kMeans to create clusters. KMeans is a machine learning algorithm that creates clusters of data points based on their features. In this case, I used the three performance metrics as features to create clusters of players with similar performance profiles.

The question then comes as we don't know how many clusters we want to have. There are two ways we can determine the optimal number of clusters, respectively the elbow method and silhouette score. The elbow method is a graphical representation of the sum of squared distances between data points and their respective cluster centroids. The silhouette score is a measure of how similar an object is to its own cluster compared to other clusters. A higher silhouette score indicates that the object is well matched to its own cluster and poorly matched to neighboring clusters.

## Finding the optimal number of clusters

For the elbow method, WCSS measures the sum of squared distances between each point and its assigned cluster center. As the WCSS value goes down, the clusters become more well-defined. The elbow point, the "knee", is the point where the WCSS value starts to decrease at a slower rate. This indicates that adding more clusters does not significantly improve the clustering quality.

On the elbow chart, we can see a sharp drop in WCSS when we add 1 more cluster to the model from k=1. In general, adding clusters always causes over-fitting, so the goal is to have the least amount of clusters as possible while explaining the variance. In this case, the "knee" of the chart is at k=2, meaning that when we have 2 clusters, the WCSS is relatively low and the model stays away from over-fitting.

![](elbow_method.png)

 The silhouette score is a measure of how similar an object is to its own cluster compared to other clusters. A higher silhouette score indicates that the object is well matched to its own cluster and poorly matched to neighboring clusters. In this case, the silhouette score is highest at k=2, which means that the clusters are well separated and the players within each cluster are similar to each other. This confirms our previous finding that k=2 is the optimal number of clusters.

![](sil_score.png)

## Reflection on the finding of the optimal number of clusters

However, **The objective of this project is to locate players who perform well but are not as costly as the super stars. If we use 2 clusters, we will only have 2 groups of players: the super stars who exceed in all three metrics and the others who have mediocre performance. We want a middle group of players who are not super stars but are still good enough to be considered. Therefore, we will use 3 clusters instead of 2.**

In [ ]:
#cluster_per_selected = performance[['PTS','eFG%','MP']]
#kmeans_obj_performance_selected = KMeans(n_clusters=3, random_state=1).fit(cluster_per_selected)
#performance['cluster_real'] = kmeans_obj_performance_selected.labels_

#fig = px.scatter_3d(
#    performance,
#    x="MP", y="eFG%", z="PTS",
#    color='cluster_real', 
#    size = "Salary",
#    hover_data=['Player','Salary'],
#    title="Minutes Played vs. Efficient FG% vs. Points Scored (Size by Salary)")
#fig.show(renderer="vscode")

I selected PTS, eFG%, and MP as the performance metrics. The color is based on the cluster that the player belongs to.

![](wosalary.png)

## Combining the clusters with salary

After confirming that we need 3 clusters, we can then create the model. Now we have three clusters of players, with high performance, medium performance and low performance. We can then combine the clusters with the salary data to see how much each player is paid compared to their performance.

## Results

In [ ]:
#cluster_per_selected = performance[['PTS','eFG%','MP']]
#kmeans_obj_performance_selected = KMeans(n_clusters=3, random_state=1).fit(cluster_per_selected)
#performance['cluster_real'] = kmeans_obj_performance_selected.labels_

#fig = px.scatter_3d(
#    performance,
#    x="MP", y="eFG%", z="PTS",
#    color='cluster_real', 
#    size = "Salary",
#    size_max=20,
#    hover_data=['Player','Salary'],
#    title="Minutes Played vs. Efficient FG% vs. Points Scored (Size by Salary)")
#fig.show(renderer="vscode")

I use salary as the size of the data points in the cluster. Each cluster has a different color and the size of the data points represents the salary of the player. The larger the data point, the higher the salary. The players in the high performance cluster are represented by blue dots (cluster 2), while the players in the medium performance cluster are represented by yellow dots (cluster 1) and those in the low performance cluster are represented by purple dots (cluster 0).

Cluster 0 is for the bench players who don't get a lot of minutes, points, and shoot poorly. Cluster 1 is for rotation players who play a moderate amount of time every game, have decent shooting stats and efficiency. Cluster 2 is for the star players with high usage and high efficiency. The star players are no doubt expensive, and the bench players are not our priority. Thus, among these players, we need to find players who are in cluster 1 but have a low salary.

![](wsalary.png)

Here I merged the salary stats into the dataset and set the size of the dots to indicate the players' salary. Here are two ways to find our ideal players:

1. Eyeballing: The 3D scatterplot has hover data where I can see the player's name and his salary. I picked a few players that belong to Cluster 1 (the yellow cluster) and went for dots with smaller sizes. From Cluster 1, the players I found are Spencer Dwinddle, Max Strus, and Robert Covington, whose salaries are $16,892,857, $20,171,427 and $12,307,692. Even though the top 25% of players' salary distribution is at $12,600,000, their salary is pretty cost-efficient compared with players like Stephen Curry with a yearly salary of $48,070,014. If we can't get these players, players like Jalen Smith, Thaddeus Young, and Jaden Ivey are good options as they still have a relatively decent performance and have medium salaries.

2. Regression Model: since we have the performance metrics and the salary, we can build a regression model to predict the salary based on the performance metrics. After we acquire the prediction, we can compare it with the player's true salary. Those who have a big positive gap between the predicted salary and the true salary are the players we should pay attention to. Those who have a big negative gap are the players we should avoid. Those whose predicted salary is close to the true salary are the players we can choose if we can't choose the good players.

The "Residual" column in the table below shows the difference between the predicted salary and the true salary. A positive residual indicates that the player is underpaid, while a negative residual indicates that the player is overpaid. The players with the largest positive residuals are the ones that we should consider acquiring. The players with the largest negative residuals are the ones that we should avoid. The players with the smallest residuals are the ones that we can consider if we can't get the good players.

### Underpaid

| Player              |     Salary | Predicted Salary |     Residual |
|---------------------|-----------:|-----------------:|-------------:|
| Tyrese Haliburton   |   4,215,120 |        17,463,136 |   –13,248,016 |
| Justin Holiday      |   6,292,440 |        17,463,136 |   –11,170,696 |
| Eric Bledsoe        |   1,300,000 |        10,931,890 |    –9,631,890 |
| Austin Reaves       |   1,563,518 |        10,931,890 |    –9,368,372 |

### Fair (close to predicted)

| Player            |     Salary | Predicted Salary |   Residual |
|-------------------|-----------:|-----------------:|-----------:|
| Norman Powell     |  16,758,621 |        17,463,136 |    –704,515 |
| Derrick White     |  16,892,857 |        17,463,136 |    –570,279 |
| P.J. Tucker       |  10,490,000 |        10,931,890 |    –441,890 |
| Cade Cunningham   |  10,552,800 |        10,931,890 |    –379,090 |

### Overpaid

| Player             |     Salary | Predicted Salary |    Residual |
|--------------------|-----------:|-----------------:|------------:|
| Stephen Curry      |  48,070,014 |        10,931,890 |   37,138,124 |
| Russell Westbrook  |  47,063,478 |        10,931,890 |   36,131,588 |
| LeBron James       |  44,474,988 |        10,931,890 |   33,543,098 |
| Kevin Durant       |  44,119,845 |        10,931,890 |   33,187,955 |


Judging from the regression model, we should pick Haliburton, Holiday, Bledsoe, and Reaves. If these players aren't available, we should pick players like Powell, White, Tucker, or Cunningham.

## Conclusion

This analysis used k-Means clustering to explore the relationship between NBA players' performance and salary. We used elbow method and silhouette score to determine the optimal amount of clusters. It was supposed to be 2 but we want to analyze players who are overpaid, fairly paid and underpaid, we set the cluster number to be 3. By focusing on key performance metrics **MP**, **eFG%**, and **PTS**, we identified three distinct player clusters: one representing high-performing "star players", one other for rotation players and another for deep bench players. The clustering highlighted undervalued players like Tyrese Haliburton, Spencer Dinwiddie and Derrick White, whose outputs suggest they deliver strong value relative to their salaries. If we cannot get these players, we should consider players like Jalen Smith, Thaddeus Young, and Jaden Ivey.